# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
pip install beautifulsoup4 schedule rich requests ipywidgets # установка библиотек, если ещё не

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import re
import schedule
import json
import sys
import json
from datetime import datetime, timedelta
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from rich.console import Console
from rich.live import Live
from rich.text import Text
from pathlib import Path
from functools import wraps
from requests.adapters import HTTPAdapter

## Задание 1. Сбор данных об одной книге (20 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
class ScraperException(Exception):
    """Base exception for the scraper."""
    pass


def _create_session() -> requests.Session:
    """Create configured HTTP session with retry strategy."""
    session = requests.Session()

    adapter = HTTPAdapter(
        pool_connections=10,
        pool_maxsize=10,
        max_retries=3
    )
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    return session

# Constants
SESSION = _create_session()
HTML_PARSER = 'html.parser'
BASE_URL = "https://books.toscrape.com/catalogue/"


def _check_valid_http_url(url: str) -> None:
    """
    Validate HTTP/HTTPS URL format.

    Args:
        url: URL string to validate

    Raises:
        ScraperException: If URL format is invalid
    """
    try:
        result = urlparse(url)
        is_valid = all([
            result.scheme in ['http', 'https'],
            result.netloc
        ])
        if not is_valid:
            raise ScraperException(f'Incorrect URL: {url}')
    except Exception as e:
        raise ScraperException(f'Incorrect URL: {url}') from e    

In [4]:
def get_book_data(book_url: str) -> dict:
    """
    Extract book data from individual book page.

    Collects title, price, rating, stock count, description,
    and product information from the table.

    Args:
        book_url: URL of the book page

    Returns:
        Dictionary with book data
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    _check_valid_http_url(book_url)
    book_data = {}

    response = SESSION.get(book_url)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, HTML_PARSER)

    # Extract main product information
    product_main = soup.find(
        'div', attrs={'class': 'col-sm-6 product_main'}
    )
    book_data['title'] = product_main.find('h1').text
    book_data['price'] = product_main.find(
        'p', attrs={'class': 'price_color'}
    ).text

    # Extract stock information
    stock_info = product_main.find(
        'p', attrs={'class': 'instock availability'}
    )
    for child in stock_info.children:
        numbers = re.findall(r'\d+', child.text)
        if numbers:
            book_data['available_count'] = numbers[0]

    # Extract rating
    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    rating_element = stock_info.find_next_siblings()[0]
    book_data['rate'] = rating_map[rating_element.get('class')[1]]

    # Extract description if exists
    description_div = soup.find(
        'div', attrs={'id': 'product_description'}
    )
    if description_div:
        book_data['description'] = description_div.find_next_siblings()[0].text

    # Extract product information table
    product_table = soup.find(
        'table', attrs={'class': 'table table-striped'}
    )
    for row in product_table.find_all('tr'):
        key = row.find_next('th').text
        value = row.find_next('td').text
        book_data[key] = value

    return book_data
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [5]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': '£51.77',
 'available_count': '22',
 'rate': 3,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put 

## Задание 2. Сбор данных обо всех книгах (20 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [6]:
def _get_progress_display(total_count: int, processed_books: int) -> Text:
    """
    Create progress display for book scraping process.

    Args:
        total_count: Total number of books to process
        processed_books: Number of books already processed

    Returns:
        Text object with progress bar
    """
    progress_percent = (processed_books / total_count * 100) \
        if total_count > 0 else 0
    progress_percent = max(0, min(100, progress_percent))

    bar_length = 30
    filled_length = int(bar_length * progress_percent / 100)
    bar = '█' * filled_length + '░' * (bar_length - filled_length)

    return Text.from_markup(
        f"Processed {processed_books} books from {total_count}. "
        f"Progress: [{bar}] [yellow]{progress_percent:.1f}%[/yellow]"
    )


def _determine_total_count(html_text: str) -> int:
    """
    Determine total count of books from catalog page.

    Args:
        html_text: HTML content of the catalog page

    Returns:
        Total number of books

    Raises:
        ScraperException: If total count cannot be determined
    """
    try:
        soup = BeautifulSoup(html_text, HTML_PARSER)
        div_tags = soup.find('div', class_='col-sm-8 col-md-9')
        form_tag = div_tags.find('form', class_='form-horizontal')
        count_text = form_tag.find_next('strong').text
        return int(count_text)
    except (AttributeError, ValueError) as e:
        raise ScraperException('Cannot determine total book count') from e


def _save_to_file(books_data: list[dict], file_path: Path) -> None:
    """
    Save book data as JSON to file.

    Args:
        books_data: List of book dictionaries
        file_path: Path to save file
    """
    if not books_data:
        return
    with open(file_path, 'w', encoding='utf-8') as file:
        for book in books_data:
            json.dump(book, file, ensure_ascii=False, indent=2)
            file.write('\n')


def timer(func):
    """
    Decorator to measure and print function execution time.

    Args:
        func: Function to measure execution time for

    Returns:
        function: Wrapped function with timing functionality
    """

    @wraps(func)
    def wrapper(*args, **kwargs):
        """
        Measure execution time of the wrapped function.

        Args:
            *args: Positional arguments for the function
            **kwargs: Keyword arguments for the function

        Returns:
            The result of the wrapped function
        """
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        execution_time = end_time - start_time
        print(
            f"Function '{func.__name__}' executed in "
            f"{execution_time:.0f} seconds"
        )
        return result

    return wrapper

In [7]:
@timer
def scrape_books(is_save: bool = True) -> list[dict]:
    """
    Scrape books from all catalog pages.

    Iterates through all pages of the catalog and parses book data.
    Optionally saves results to a file.

    Args:
        is_save: Flag to save results to file, defaults to True

    Returns:
        List of dictionaries with book data
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    page_url = BASE_URL + '/page-1.html'
    response = SESSION.get(page_url)
    total_count = _determine_total_count(response.text)
    list_books = []
    console = Console(file=sys.stderr, force_terminal=False)

    with Live(
            _get_progress_display(total_count, len(list_books)),
            refresh_per_second=1, console=console, transient=True,
            auto_refresh=False
    ) as live:
        while response:
            soup = BeautifulSoup(response.text, HTML_PARSER)
            div_tags = soup.find(
                'div', attrs={'class': 'col-sm-8 col-md-9'}
            )
            last_link = None

            for tag in div_tags.find_all('a'):
                href = tag.get('href')
                if (last_link and href == last_link or
                        tag.find_parent('li', attrs={'class': 'previous'})):
                    continue
                elif tag.find_parent('li', attrs={'class': 'next'}):
                    response = SESSION.get(
                        f'https://books.toscrape.com/catalogue/{href}'
                    )
                    break
                else:
                    last_link = href
                    (list_books.append(get_book_data(
                        f'https://books.toscrape.com/catalogue/{href}'))
                    )
            else:
                response = None
            live.update(_get_progress_display(total_count, len(list_books)))
            live.refresh()

    if is_save:
        file_path = Path("../artifacts/books_data.txt")
        file_path.parent.mkdir(parents=True, exist_ok=True)
        _save_to_file(list_books, file_path)

    return list_books
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [8]:
# Проверка работоспособности функции
res = scrape_books(False) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

Output()

Function 'scrape_books' executed in 175 seconds
<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (10 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [9]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
def _check_valid_time(time_str: str) -> None:
    """
    Check if the time string matches HH:MM format with valid values.

    Args:
        time_str: Time string to validate in HH:MM format

    Raises:
        ScraperException: If time format is invalid or values are out
        of range
    """
    pattern = r'^([01]?\d|2[0-3]):([0-5]\d)$'
    if not re.match(pattern, time_str):
        raise ScraperException(
            f'Incorrect time format: {time_str}, should be HH:MM'
        )


def _get_scheduled_display(scraper_time: str) -> Text:
    """
    Create progress display for scheduled scraping.

    Args:
        scraper_time: Scheduled time in HH:MM format

    Returns:
        Text object with progress information
    """
    now: datetime = datetime.now()
    target_time: datetime = now.replace(
        hour=int(scraper_time.split(':')[0]),
        minute=int(scraper_time.split(':')[1]),
        second=0,
        microsecond=0
    )

    # if time has passed today, schedule for tomorrow
    if target_time <= now:
        target_time += timedelta(days=1)

    time_left: timedelta = target_time - now
    total_seconds: float = time_left.total_seconds()

    # format time display
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return Text.from_markup(
        f"The scraping is daily scheduled for [bold green]"
        f"{scraper_time}[/bold green]. The next launch is in: "
        f"[cyan]{int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}[/cyan]"
    )        


def scrape_books_at_time(scraper_time: str) -> None:
    """
    Schedule book scraping at specified time.

    Args:
        scraper_time: Time in HH:MM format for daily execution
    """
    _check_valid_time(scraper_time)
    schedule.every().day.at(scraper_time).do(scrape_books)
    console = Console(file=sys.stderr, force_terminal=False)
    live_stopped = False

    with Live(
            _get_scheduled_display(scraper_time), refresh_per_second=1,
            console=console, transient=True, auto_refresh=False
    ) as live:
        while True:
            if schedule.idle_seconds() < 1:
                live.stop()
                live_stopped = True
            schedule.run_pending()
            time.sleep(1)
            if live_stopped and schedule.idle_seconds() > 1:
                live.start()
                live_stopped = False
            if not live_stopped:
                live.update(_get_scheduled_display(scraper_time))
                live.refresh()
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [10]:
# scrape_books_at_time('12:00')

## Задание 4. Написание автотестов (15 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [11]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
import sys

!{sys.executable} -m pytest /Users/Marya/Desktop/books_scraper/tests/test_scraper.py -v

]9;4;3;\============================= test session starts ==============================
platform darwin -- Python 3.13.7, pytest-9.0.0, pluggy-1.6.0 -- /opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/bin/python
cachedir: .pytest_cache
rootdir: /Users/Marya/Desktop/books_scraper
plugins: anyio-4.10.0
collected 13 items                                                             

../tests/test_scraper.py::test_get_book_data_invalid_url[ssh://server.com] ]9;4;1;0\PASSED [  7%]
../tests/test_scraper.py::test_get_book_data_invalid_url[http://] ]9;4;1;7\PASSED [ 15%]
../tests/test_scraper.py::test_get_book_data_invalid_url[https://] ]9;4;1;15\PASSED [ 23%]
../tests/test_scraper.py::test_get_book_data_invalid_url[books.toscrape.com] ]9;4;1;23\PASSED [ 30%]
../tests/test_scraper.py::test_get_book_data_invalid_url[htts://books.toscrape.com] ]9;4;1;30\PASSED [ 38%]
../tests/test_scraper.py::test_scrape_books_at_time_invalid_time[19:] ]9;4;1;38\PASSED [ 46%]
../tests/test_scraper.py

## Задание 5. Оформление проекта на GitHub и работа с Git (35 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds_2025.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```